# Semaine 2 & 3 — Fine-tuning SFT (LoRA) et Alignement DPO

**Objectif** : Spécialiser le modèle Qwen3-1.7B-Base sur notre corpus médical via :
1. **SFT (Supervised Fine-Tuning)** avec LoRA — adaptation efficace sur les 5 000 paires instruction-réponse
2. **DPO (Direct Preference Optimization)** — alignement sur les préférences cliniques

**Environnement** : Kaggle Notebook avec GPU T4 (15 Go VRAM)

**Prérequis** : Avoir ajouté le dataset `chsa-medical-data` à ce notebook (bouton *Add Data*)

In [ ]:
# Installation des dépendances
!pip install -q torch transformers accelerate peft trl datasets bitsandbytes pyyaml matplotlib wandb

In [ ]:
import json
import logging
import os
from pathlib import Path

import torch
import yaml
from datasets import load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# === Détection de l'environnement ===
if Path("/kaggle/input").exists():
    ENV = "kaggle"
    DATA_ROOT = Path("/kaggle/input/chsa-medical-data")
    OUTPUT_ROOT = Path("/kaggle/working")
elif Path("/content").exists():
    ENV = "colab"
    DATA_ROOT = Path("/content/chsa-medical-data")
    OUTPUT_ROOT = Path("/content")
else:
    ENV = "local"
    DATA_ROOT = Path.cwd().parent / "data"
    OUTPUT_ROOT = Path.cwd().parent

print(f"Environnement : {ENV}")
print(f"Données : {DATA_ROOT}")
print(f"Sorties : {OUTPUT_ROOT}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'}")
if torch.cuda.is_available():
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} Go")

In [ ]:
# === Vérification des fichiers ===
expected_files = [
    DATA_ROOT / "sft" / "train.jsonl",
    DATA_ROOT / "sft" / "eval.jsonl",
    DATA_ROOT / "dpo" / "train.jsonl",
    DATA_ROOT / "dpo" / "eval.jsonl",
    DATA_ROOT / "configs" / "sft_config.yaml",
    DATA_ROOT / "configs" / "dpo_config.yaml",
]

all_ok = True
for f in expected_files:
    status = "OK" if f.exists() else "MANQUANT"
    if not f.exists():
        all_ok = False
    print(f"  {status:10s} {f}")

if not all_ok:
    print("\n>>> Sur Kaggle : cliquez 'Add Data' et ajoutez le dataset 'chsa-medical-data'")
    print(">>> Sur Colab  : uploadez le dossier chsa-medical-data/ dans /content/")

In [ ]:
# === Chargement de la configuration SFT ===
with open(DATA_ROOT / "configs" / "sft_config.yaml") as f:
    sft_config = yaml.safe_load(f)

# Adapter les paths pour l'environnement cloud
sft_config["data"]["train_file"] = str(DATA_ROOT / "sft" / "train.jsonl")
sft_config["data"]["eval_file"] = str(DATA_ROOT / "sft" / "eval.jsonl")
sft_config["training"]["output_dir"] = str(OUTPUT_ROOT / "outputs" / "sft")

# Désactiver wandb si pas configuré
if "WANDB_API_KEY" not in os.environ:
    sft_config["training"]["report_to"] = "none"
    print("WandB non configuré — logging désactivé")

print(f"Modèle : {sft_config['model']['name']}")
print(f"LoRA r={sft_config['lora']['r']}, alpha={sft_config['lora']['lora_alpha']}")
print(f"Epochs : {sft_config['training']['num_train_epochs']}")
print(f"Batch size : {sft_config['training']['per_device_train_batch_size']}")
print(f"Learning rate : {sft_config['training']['learning_rate']}")

---
## Partie 1 : Fine-tuning supervisé (SFT) avec LoRA

### 1.1 Vérification des données d'entraînement

In [ ]:
# Chargement des datasets SFT
sft_train = load_dataset("json", data_files=sft_config["data"]["train_file"], split="train")
sft_eval = load_dataset("json", data_files=sft_config["data"]["eval_file"], split="train")

print(f"SFT Train : {len(sft_train)} exemples")
print(f"SFT Eval  : {len(sft_eval)} exemples")
print(f"Colonnes : {sft_train.column_names}")
print(f"\nExemple (premier message user) :")
print(sft_train[0]["messages"][1]["content"][:300])

In [ ]:
# Distribution des sources et langues
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

sources = Counter(sft_train["source"])
langues = Counter(sft_train["langue"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = [s.split("/")[-1] for s in sources.keys()]
axes[0].bar(labels, sources.values(), color=plt.cm.Set2(np.linspace(0, 1, len(sources))))
axes[0].set_title("Distribution par source (SFT train)")
axes[0].set_ylabel("Nombre d'exemples")
for i, v in enumerate(sources.values()):
    axes[0].text(i, v + 20, str(v), ha="center", fontsize=9)

axes[1].pie(langues.values(), labels=langues.keys(), autopct="%1.0f%%",
            colors=["#66b3ff", "#ff9999"], startangle=90)
axes[1].set_title("Distribution par langue (SFT train)")

plt.tight_layout()
plt.show()

### 1.2 Chargement du modèle de base et configuration LoRA

In [ ]:
# Chargement du modèle Qwen3-1.7B-Base
model_name = sft_config["model"]["name"]
dtype = getattr(torch, sft_config["model"]["torch_dtype"])

print(f"Chargement de {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Modèle chargé : {model_name}")
print(f"Paramètres totaux : {model.num_parameters():,}")
print(f"dtype : {dtype}")

In [ ]:
# Configuration LoRA
lora_cfg = sft_config["lora"]

peft_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    task_type=TaskType.CAUSAL_LM,
)

print("=== Configuration LoRA ===")
print(f"Rang (r) : {peft_config.r}")
print(f"Alpha : {peft_config.lora_alpha}")
print(f"Dropout : {peft_config.lora_dropout}")
print(f"Modules ciblés : {peft_config.target_modules}")
print(f"Ratio alpha/r = {peft_config.lora_alpha / peft_config.r:.1f}")

### 1.3 Test du modèle de base (avant fine-tuning)

On teste le modèle de base sur quelques questions médicales pour avoir un point de comparaison.

In [ ]:
# Prompts de test — cas de triage représentatifs
test_prompts = [
    "Un patient de 55 ans se présente aux urgences avec une douleur thoracique aiguë irradiant vers le bras gauche. Quel est le niveau de priorité ?",
    "Une femme de 30 ans consulte pour un mal de gorge depuis 2 jours avec une légère fièvre à 38.2°C. Quel est le niveau de priorité ?",
    "Un enfant de 3 ans est amené pour une éruption cutanée diffuse apparue il y a 1 heure après avoir mangé des cacahuètes. Il présente un gonflement des lèvres. Quel est le niveau de priorité ?",
]

def generate_responses(model, tokenizer, prompts, label=""):
    """Génère des réponses pour une liste de prompts."""
    model.eval()
    responses = []
    for i, prompt in enumerate(prompts):
        messages = [
            {"role": "system", "content": "Vous etes un assistant medical specialise dans le triage des urgences."},
            {"role": "user", "content": prompt},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=1.0,
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        responses.append(response)
        print(f"\n--- Cas {i+1} ---")
        print(f"Q: {prompt[:100]}...")
        print(f"R: {response[:500]}")
    return responses

print("=== Réponses du modèle de BASE (avant SFT) ===")
print("=" * 60)
base_responses = generate_responses(model, tokenizer, test_prompts, "BASE")

### 1.4 Lancement de l'entraînement SFT

In [ ]:
# Configuration du SFTTrainer
training_cfg = sft_config["training"]

sft_training_args = SFTConfig(
    output_dir=training_cfg["output_dir"],
    num_train_epochs=training_cfg["num_train_epochs"],
    per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=training_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
    learning_rate=training_cfg["learning_rate"],
    weight_decay=training_cfg["weight_decay"],
    warmup_ratio=training_cfg["warmup_ratio"],
    lr_scheduler_type=training_cfg["lr_scheduler_type"],
    logging_steps=training_cfg["logging_steps"],
    eval_strategy=training_cfg["eval_strategy"],
    eval_steps=training_cfg["eval_steps"],
    save_strategy=training_cfg["save_strategy"],
    save_steps=training_cfg["save_steps"],
    save_total_limit=training_cfg["save_total_limit"],
    bf16=training_cfg["bf16"],
    max_seq_length=training_cfg["max_seq_length"],
    gradient_checkpointing=training_cfg["gradient_checkpointing"],
    report_to=training_cfg["report_to"],
    seed=training_cfg.get("seed", 42),
)

trainer = SFTTrainer(
    model=model,
    args=sft_training_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()

In [ ]:
# Lancement de l'entraînement SFT
print("Démarrage de l'entraînement SFT...")
sft_results = trainer.train()
print(f"\nEntraînement terminé !")
print(f"Loss finale : {sft_results.training_loss:.4f}")

In [ ]:
# Sauvegarde du modèle SFT final
sft_final_path = str(OUTPUT_ROOT / "outputs" / "sft" / "final")
trainer.save_model(sft_final_path)
tokenizer.save_pretrained(sft_final_path)
print(f"Modèle SFT sauvegardé dans : {sft_final_path}")

### 1.5 Visualisation des métriques SFT

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval loss", marker="o", markersize=4)

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("SFT — Courbes de loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if eval_losses:
    print(f"Meilleure eval loss : {min(l for _, l in eval_losses):.4f}")

### 1.6 Test du modèle SFT (après fine-tuning)

In [ ]:
print("=== Réponses du modèle SFT (après fine-tuning) ===")
print("=" * 60)
sft_responses = generate_responses(trainer.model, tokenizer, test_prompts, "SFT")

In [ ]:
# Comparaison Base vs SFT
print("=" * 80)
print("COMPARAISON : Modèle de base vs Modèle SFT")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 80}")
    print(f"CAS {i+1}: {prompt[:100]}...")
    print(f"{'─' * 80}")
    print(f"\n[BASE] {base_responses[i][:300]}")
    print(f"\n[SFT]  {sft_responses[i][:300]}")

---
## Partie 2 : Alignement par préférences (DPO)

On prend le modèle SFT fine-tuné et on l'aligne avec DPO sur les paires chosen/rejected d'UltraMedical-Preference.

### 2.1 Préparation

In [ ]:
# Libérer la mémoire GPU du SFT
import gc

del trainer, model, sft_train, sft_eval
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM libre : {(torch.cuda.mem_get_info()[0] / 1024**3):.2f} Go")

from trl import DPOConfig, DPOTrainer

# Chargement de la config DPO
with open(DATA_ROOT / "configs" / "dpo_config.yaml") as f:
    dpo_config = yaml.safe_load(f)

# Adapter les paths
dpo_config["model"]["name"] = sft_final_path  # Pointe vers le SFT qu'on vient d'entraîner
dpo_config["data"]["train_file"] = str(DATA_ROOT / "dpo" / "train.jsonl")
dpo_config["data"]["eval_file"] = str(DATA_ROOT / "dpo" / "eval.jsonl")
dpo_config["training"]["output_dir"] = str(OUTPUT_ROOT / "outputs" / "dpo")

# Overrides mémoire pour T4 15 Go
dpo_config["training"]["per_device_train_batch_size"] = 1
dpo_config["training"]["per_device_eval_batch_size"] = 1
dpo_config["training"]["gradient_accumulation_steps"] = max(
    dpo_config["training"].get("gradient_accumulation_steps", 1) * 4, 8
)
dpo_config["training"]["max_length"] = 1024
dpo_config["training"]["max_prompt_length"] = 512
dpo_config["training"]["gradient_checkpointing"] = True

if "WANDB_API_KEY" not in os.environ:
    dpo_config["training"]["report_to"] = "none"

print(f"Modèle SFT : {dpo_config['model']['name']}")
print(f"Beta (DPO) : {dpo_config['training']['beta']}")
print(f"Learning rate : {dpo_config['training']['learning_rate']}")
print(f"Batch effectif : {dpo_config['training']['per_device_train_batch_size']} x {dpo_config['training']['gradient_accumulation_steps']} grad accum")
print(f"max_length={dpo_config['training']['max_length']}, max_prompt_length={dpo_config['training']['max_prompt_length']}")


### 2.2 Chargement du modèle SFT et des données DPO

In [ ]:
# Chargement du modèle base en 4-bit puis empilage de l'adaptateur SFT
from transformers import BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

dtype = getattr(torch, dpo_config["model"]["torch_dtype"])
base_model_name = sft_config["model"]["name"]

print(f"Base : {base_model_name}")
print(f"Adaptateur SFT : {sft_final_path}")

tokenizer = AutoTokenizer.from_pretrained(sft_final_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
base.config.use_cache = False

# Empile l'adaptateur SFT en mode entraînable — DPO continuera de l'entraîner
dpo_model = PeftModel.from_pretrained(base, sft_final_path, is_trainable=True)
dpo_model.print_trainable_parameters()


In [ ]:
# Chargement des données DPO
dpo_train = load_dataset("json", data_files=dpo_config["data"]["train_file"], split="train")
dpo_eval = load_dataset("json", data_files=dpo_config["data"]["eval_file"], split="train")

# Conversion au format conversationnel attendu par TRL :
# - prompt (str) -> [{"role": "user", "content": prompt}]
# - chosen / rejected (liste [user, assistant]) -> [{"role": "assistant", "content": ...}]
def _to_conversational(example):
    chosen_msgs = example["chosen"]
    rejected_msgs = example["rejected"]
    assistant_chosen = next(m for m in chosen_msgs if m["role"] == "assistant")
    assistant_rejected = next(m for m in rejected_msgs if m["role"] == "assistant")
    return {
        "prompt": [{"role": "user", "content": example["prompt"]}],
        "chosen": [{"role": "assistant", "content": assistant_chosen["content"]}],
        "rejected": [{"role": "assistant", "content": assistant_rejected["content"]}],
    }

_keep = {"prompt", "chosen", "rejected"}
_remove = [c for c in dpo_train.column_names if c not in _keep]
dpo_train = dpo_train.map(_to_conversational, remove_columns=_remove)
dpo_eval = dpo_eval.map(_to_conversational, remove_columns=_remove)

print(f"DPO Train : {len(dpo_train)} exemples")
print(f"DPO Eval  : {len(dpo_eval)} exemples")
print(f"Colonnes : {dpo_train.column_names}")
print(f"\nExemple de prompt :")
print(dpo_train[0]["prompt"][0]["content"][:300])


### 2.3 Lancement de l'entraînement DPO

In [ ]:
# Configuration DPOTrainer
dpo_training_cfg = dpo_config["training"]

dpo_training_args = DPOConfig(
    output_dir=dpo_training_cfg["output_dir"],
    num_train_epochs=dpo_training_cfg["num_train_epochs"],
    per_device_train_batch_size=dpo_training_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=dpo_training_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=dpo_training_cfg["gradient_accumulation_steps"],
    learning_rate=dpo_training_cfg["learning_rate"],
    weight_decay=dpo_training_cfg["weight_decay"],
    warmup_ratio=dpo_training_cfg["warmup_ratio"],
    lr_scheduler_type=dpo_training_cfg["lr_scheduler_type"],
    logging_steps=dpo_training_cfg["logging_steps"],
    eval_strategy=dpo_training_cfg["eval_strategy"],
    eval_steps=dpo_training_cfg["eval_steps"],
    save_strategy=dpo_training_cfg["save_strategy"],
    save_steps=dpo_training_cfg["save_steps"],
    save_total_limit=dpo_training_cfg["save_total_limit"],
    bf16=dpo_training_cfg["bf16"],
    max_length=dpo_training_cfg["max_length"],
    max_prompt_length=dpo_training_cfg["max_prompt_length"],
    beta=dpo_training_cfg["beta"],
    gradient_checkpointing=dpo_training_cfg["gradient_checkpointing"],
    report_to=dpo_training_cfg["report_to"],
    seed=dpo_training_cfg.get("seed", 42),
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_training_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer,
)

dpo_trainer.model.print_trainable_parameters()

In [ ]:
# Lancement de l'entraînement DPO
print("Démarrage de l'entraînement DPO...")
dpo_results = dpo_trainer.train()
print(f"\nEntraînement DPO terminé !")
print(f"Loss finale : {dpo_results.training_loss:.4f}")

In [ ]:
# Sauvegarde du modèle DPO final
dpo_final_path = str(OUTPUT_ROOT / "outputs" / "dpo" / "final")
dpo_trainer.save_model(dpo_final_path)
tokenizer.save_pretrained(dpo_final_path)
print(f"Modèle DPO sauvegardé dans : {dpo_final_path}")

### 2.4 Visualisation des métriques DPO

In [ ]:
import matplotlib.pyplot as plt

dpo_log_history = dpo_trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in dpo_log_history if "loss" in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in dpo_log_history if "eval_loss" in h]
rewards_chosen = [(h["step"], h["rewards/chosen"]) for h in dpo_log_history if "rewards/chosen" in h]
rewards_rejected = [(h["step"], h["rewards/rejected"]) for h in dpo_log_history if "rewards/rejected" in h]
reward_margins = [(h["step"], h["rewards/margins"]) for h in dpo_log_history if "rewards/margins" in h]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, label="Train loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    axes[0].plot(steps, losses, label="Eval loss", marker="o", markersize=4)
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("DPO — Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rewards
if rewards_chosen:
    steps, rewards = zip(*rewards_chosen)
    axes[1].plot(steps, rewards, label="Chosen", color="green")
if rewards_rejected:
    steps, rewards = zip(*rewards_rejected)
    axes[1].plot(steps, rewards, label="Rejected", color="red")
axes[1].set_xlabel("Steps")
axes[1].set_ylabel("Reward")
axes[1].set_title("DPO — Rewards (chosen vs rejected)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reward margin
if reward_margins:
    steps, margins = zip(*reward_margins)
    axes[2].plot(steps, margins, label="Margin", color="purple")
    axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[2].set_xlabel("Steps")
axes[2].set_ylabel("Margin")
axes[2].set_title("DPO — Reward margin (chosen - rejected)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5 Test du modèle DPO (après alignement)

In [ ]:
print("=== Réponses du modèle DPO (après alignement) ===")
print("=" * 60)
dpo_responses = generate_responses(dpo_trainer.model, tokenizer, test_prompts, "DPO")

---
## Partie 3 : Comparaison finale Base vs SFT vs DPO

In [ ]:
print("=" * 100)
print("COMPARAISON FINALE : Base vs SFT vs DPO")
print("=" * 100)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'━' * 100}")
    print(f"CAS {i+1}: {prompt}")
    print(f"{'━' * 100}")
    print(f"\n[BASE]")
    print(base_responses[i][:400])
    print(f"\n[SFT]")
    print(sft_responses[i][:400])
    print(f"\n[DPO]")
    print(dpo_responses[i][:400])
    print()

---
## Sauvegarde des artefacts pour téléchargement

Sur Kaggle, les fichiers dans `/kaggle/working/` sont téléchargeables après exécution.

In [ ]:
# Résumé des fichiers produits
import os

print("=== Artefacts produits ===")
for root, dirs, files in os.walk(str(OUTPUT_ROOT / "outputs")):
    level = root.replace(str(OUTPUT_ROOT / "outputs"), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"{sub_indent}{file} ({size_mb:.1f} Mo)")

---
## Résumé

| Phase | Modèle | Description |
|-------|--------|-------------|
| Base | Qwen3-1.7B-Base | Modèle pré-entraîné, pas de spécialisation médicale |
| SFT | + LoRA fine-tuning | Spécialisé sur 5 000 paires médicales (FR/EN) |
| DPO | + Alignement préférences | Aligné sur les pratiques cliniques validées |

**Fichiers produits :**
- `outputs/sft/final/` — Modèle SFT (adaptateurs LoRA + tokenizer)
- `outputs/dpo/final/` — Modèle DPO final (adaptateurs LoRA + tokenizer)

**Prochaine étape :** Évaluation complète et déploiement (Semaine 4)